In [ ]:
import urllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# A quick detour on 'f-strings'
In python there is a really nice way to insert values 
from variables into a string variable. This task is 
referred to in programming as string formatting. In the
past there were other ways to do this in python, but 
a few years ago they added this approach which is generally
seen as the best way to do this. For an overview see this:
https://realpython.com/python-f-strings/


In [ ]:
random_numbers = np.random.random(5)
print('The average of my random numbers is', np.mean(random_numbers))
print(f'The average of my random numbers is {np.mean(random_numbers):0.2f}')


# Access streamflow data for Niagara River!
With that out of the way let's start to work towards being able to grab data on the fly from the USGS website. And now we've defined the site id for the Verde River, as well as some start and end dates to get the data for. With those defined clearly it makes it much easier for someone else to understand what you are trying to do.

# Practice 1: Apply for your API KEY

It is highly recommended to have an API key to access the USGS code.

In [ ]:
# please apply your own API KEY here: https://api.waterdata.usgs.gov/signup/
# Your API Key needs to be input as a string
API_KEY = 'YOUR_API_KEY_HERE'

## 1. define site specific information

## let's download data for `Water Year 2023`
Water Year 2023 ranges from October 1, 2022 to September 30, 2023

In [ ]:
args = {
    'monitoring_location_id': 'USGS-04216000',
    'parameter_code': '00060',              # discharge, cfs
    'statistic_id': '00003',                # daily mean
    'datetime': '2022-10-01/2023-09-30',
    'f': 'csv',
    'limit': 10000,
}

In [ ]:
query = urllib.parse.urlencode(args, safe='/')   # safe='/' keeps the date range intact

In [ ]:
query

## 2. Create the url and access the data using `urllib`

Now we can use f-strings to insert these values into the query URL which will point to the same website that we saw in the lecture portion
You can verify this by copying the URL into your web browser.

In [ ]:
niagara_url = f'https://api.waterdata.usgs.gov/ogcapi/v1/collections/daily/items?{query}'
print(niagara_url)

## 3. Read the data using `pandas`

In [ ]:
# With that we need to download the data and get it into pandas.
# To download the data we'll use the `urllib` module which is 
# built into the python "standard library" of stuff you get for
# free when you install python. We use the `urllib.request.urlopen`
# function which simply opens a connection to the url, just like 
# going to the url in your web browser. Then, we can put the `response`
# into `pd.read_table`. There are a lot of other parameters going 
# into this function now, and this is very common for when you scrape
# data directly from the internet because formats vary.

request = urllib.request.Request(niagara_url, headers={'X-Api-Key': API_KEY})
response = urllib.request.urlopen(request)

#  read the table
df = pd.read_csv(response)
df.head()

# 4. Postprocess data
## The current dataframe uses digital number as the index, which is not very practical. 

In [ ]:
# Reshape the raw API response into a tidy, time-indexed table.
# Each method returns a new DataFrame, so they chain left to right;
# the outer parentheses just let us break the chain across lines.
df = (
    df
    # Give the API's column names ones that mean something in a hydrology
    # notebook. Anything not listed here is left alone.
    .rename(columns={'time': 'date',
                     'value': 'streamflow',
                     'approval_status': 'quality_flag'})
    # Dates arrive as plain strings. Convert to real datetimes so we can
    # slice by date range and resample later.
    # (lambda d: ... means "the DataFrame as it exists at this point in
    #  the chain" — we need the renamed version, not the original df.)
    .assign(date=lambda d: pd.to_datetime(d['date']))
    # Move the dates out of a column and into the index. This is what makes
    # it a time series: df.loc['2023-01'] and df.resample('ME') now work.
    .set_index('date')
    # Sort oldest to newest. Never assume the API returns rows in order,
    # and most time-series operations expect a sorted index.
    .sort_index()
)

## the code above is super complex, but it is actually equivalent to the code below

In [ ]:
# 0. reread table
request = urllib.request.Request(niagara_url, headers={'X-Api-Key': API_KEY})
response = urllib.request.urlopen(request)
df = pd.read_csv(response)

# 1. Rename the API's columns to names that mean something here.
df = df.rename(columns={'time': 'date',
                        'value': 'streamflow',
                        'approval_status': 'quality_flag'})

# 2. Convert the date strings to real datetimes.
df['date'] = pd.to_datetime(df['date'])

# 3. Move dates from a column into the index — this makes it a time series.
df = df.set_index('date')

# 4. Sort oldest to newest; never assume the API returns rows in order.
df = df.sort_index()

# 5. Quick visualization

In [ ]:
# We can quickly take a look at the time series of the data
df['streamflow'].plot()

## Based on the plot above, we can observe that there are flood events at the end of December 2022. Why was there a flood?

# 6. Practice 2: Can you modify the code to download data for `Water Year 2024`?